# B3 ViSEC Exploratory Data Analysis and SQL

This notebook reproduces the metadata EDA required after B2. It uses all B2-eligible utterances for descriptive analysis and uses the frozen split labels only to audit coverage and leakage risk. No model, scaler, feature selector, or test-driven decision is fitted here.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

def show_figure(path):
    image = plt.imread(path)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(image)
    ax.axis('off')
    plt.show()

## Rebuild the B3 evidence

The pipeline creates a local SQLite database, executes four named read-only queries, exports their result tables, and regenerates the three EDA figures. The SQLite file is generated data and is intentionally ignored by Git.

In [ ]:
manifest_path = ROOT / 'data/manifests/split_manifest.csv'
database_path = ROOT / 'data/processed/visec_metadata.sqlite'
sql_path = ROOT / 'sql/B3_rq_queries.sql'
table_dir = ROOT / 'reports/tables/b3'
figure_dir = ROOT / 'reports/figures/b3'

completed = subprocess.run(
    [sys.executable, str(ROOT / 'src/fedecai/eda/b3_eda_sql.py')],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
summary = json.loads((table_dir / 'b3_summary.json').read_text(encoding='utf-8'))
summary

## Univariate EDA

Duration is summarized before any fixed-length preprocessing. The main panel is capped at the 99th percentile so the dense part of the distribution stays legible; the annotation retains the long-tail information. A duration cap must later be selected from training data only.

In [ ]:
show_figure(figure_dir / 'b3_duration_distribution.png')
pd.Series(summary['duration_seconds'], name='seconds').to_frame()

## Bivariate EDA tied to RQ2

The normalized stacked bars compare emotion composition within each accent. Sample sizes are shown under the accent labels because the three groups are strongly imbalanced.

In [ ]:
show_figure(figure_dir / 'b3_emotion_composition_by_accent.png')
emotion_by_accent = pd.read_csv(table_dir / 'rq2_emotion_distribution_by_accent.csv')
emotion_by_accent

## Multivariate EDA tied to RQ1 and RQ2

The heatmap combines accent, emotion, and median duration. Differences are descriptive evidence of possible nuisance structure, not proof that duration causes accent or emotion performance differences.

In [ ]:
show_figure(figure_dir / 'b3_duration_accent_emotion_heatmap.png')
quality = pd.read_csv(table_dir / 'rq1_rq2_quality_by_accent_emotion.csv')
quality

## SQL evidence and leakage checks

The SQL file uses CTEs and window functions. Speaker concentration is reported because row-level counts can be dominated by a small number of people. The full split grid also exposes zero-count accent-emotion cells instead of silently dropping them.

In [ ]:
speaker_concentration = pd.read_csv(table_dir / 'rq1_rq2_speaker_concentration.csv')
split_coverage = pd.read_csv(table_dir / 'b5_split_coverage_check.csv')
print(speaker_concentration.to_string(index=False))
print('\nZero-count split cells:')
print(split_coverage.loc[split_coverage['utterances'].eq(0)].to_string(index=False))

In [ ]:
largest = summary['largest_speaker']
print(f"Largest speaker: {largest['speaker_id']} with {largest['samples']:,} utterances ({largest['eligible_share']:.2%} of eligible data).")
print('Missing split x accent x emotion cells:', summary['missing_split_cells'])
print('Decision: do not overwrite frozen split v1; create a documented v2 before modelling if full cross-stratum test coverage is required.')

## B3 handoff

The reproducible outputs are the named SQL file, four CSV evidence tables, three figures, and the B3 report. B4 should add feature-level and advanced interactive visualizations only after the B3 split-coverage finding has been resolved or explicitly accepted.